In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.sql import SparkSession

import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("02_BronzeToSilver")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")

    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")
    .config("spark.sql.parquet.output.committer.class", "org.apache.parquet.hadoop.ParquetOutputCommitter")

    .getOrCreate()
)
spark

In [2]:
bronze = (
    spark.read
    .option("header", True)
    .csv("s3a://bronze/online_retail_II.csv")
)

print(bronze.count())

bronze.show(5)

1067371
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows



In [3]:
bronze.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)



In [4]:
bronze = bronze.withColumnRenamed(
    "Customer ID",
    "CustomerID"
)

In [5]:
silver = (
    bronze
    .withColumn("CustomerID", col("CustomerID").cast("long"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Price", col("Price").cast("double"))
    # .withColumn(
    #     "InvoiceDate",
    #     to_timestamp("InvoiceDate", "MM/d/yyyy H:mm")
    # )
)

In [6]:
silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in silver.columns
]).show()

+-------+---------+-----------+--------+-----------+-----+----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|CustomerID|Country|
+-------+---------+-----------+--------+-----------+-----+----------+-------+
|      0|        0|       4382|       0|          0|    0|    243007|      0|
+-------+---------+-----------+--------+-----------+-----+----------+-------+



In [7]:
silver = silver.filter(col("CustomerID").isNotNull())

In [8]:
silver = silver.filter(col("Description").isNotNull())

In [9]:
silver = silver.filter((col("Quantity") > 0) & (col("Price") > 0))

In [10]:
silver = silver.withColumn("IsCancelled", col("Invoice").startswith("C"))

In [11]:
silver = silver.withColumn("TotalAmount", F.round(F.col("Quantity") * F.col("Price"), 2))

In [12]:
silver.printSchema()

silver.show(10, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- IsCancelled: boolean (nullable = true)
 |-- TotalAmount: double (nullable = true)

+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|CustomerID|Country       |IsCancelled|TotalAmount|
+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085     |United Kingdom|false      |83.4       |
|489434 |79323P   |P

In [13]:
silver.select(
    "Invoice",
    "CustomerID",
    "Quantity",
    "Price",
    "IsCancelled",
    "TotalAmount"
).show(20, truncate=False)

+-------+----------+--------+-----+-----------+-----------+
|Invoice|CustomerID|Quantity|Price|IsCancelled|TotalAmount|
+-------+----------+--------+-----+-----------+-----------+
|489434 |13085     |12      |6.95 |false      |83.4       |
|489434 |13085     |12      |6.75 |false      |81.0       |
|489434 |13085     |12      |6.75 |false      |81.0       |
|489434 |13085     |48      |2.1  |false      |100.8      |
|489434 |13085     |24      |1.25 |false      |30.0       |
|489434 |13085     |24      |1.65 |false      |39.6       |
|489434 |13085     |24      |1.25 |false      |30.0       |
|489434 |13085     |10      |5.95 |false      |59.5       |
|489435 |13085     |12      |2.55 |false      |30.6       |
|489435 |13085     |12      |3.75 |false      |45.0       |
|489435 |13085     |24      |1.65 |false      |39.6       |
|489435 |13085     |12      |2.55 |false      |30.6       |
|489436 |13078     |10      |5.95 |false      |59.5       |
|489436 |13078     |18      |5.45 |false

In [14]:
print("Total Rows:", silver.count())

print(
    "Distinct Customers:",
    silver.select("CustomerID")
             .distinct()
             .count()
)

silver.groupBy("IsCancelled").count().show()

Total Rows: 805549
Distinct Customers: 5878
+-----------+------+
|IsCancelled| count|
+-----------+------+
|      false|805549|
+-----------+------+



In [15]:
(
    silver.write
    .mode("overwrite")
    .parquet(
        "s3a://silver/customer"
    )
)

In [16]:
check = spark.read.parquet(
    "s3a://silver/customer"
)

print(check.count())

check.show(5)

805549
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country|IsCancelled|TotalAmount|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----------+-----------+
| 523213|    22734|SET OF 6 RIBBONS ...|       6|2010-09-21 10:47:00| 2.55|     16945|United Kingdom|      false|       15.3|
| 523213|    22086|PAPER CHAIN KIT 5...|       6|2010-09-21 10:47:00| 2.95|     16945|United Kingdom|      false|       17.7|
| 523213|    22952|72 CAKE CASES VIN...|      24|2010-09-21 10:47:00| 0.55|     16945|United Kingdom|      false|       13.2|
| 523213|    22596|CHRISTMAS STAR WI...|      12|2010-09-21 10:47:00| 0.85|     16945|United Kingdom|      false|       10.2|
| 523213|    21821|GLITTER STAR GARL...|       6|2010-09-21 10:47:00| 3.75|     16945|United Kingdom|      fals

In [17]:
from pyspark.sql.functions import count

print("Customers:",
      check.select("CustomerID").distinct().count())

print("Invoices:",
      check.select("Invoice").distinct().count())

check.groupBy("Country") \
      .count() \
      .orderBy("count", ascending=False) \
      .show(10)

Customers: 5878
Invoices: 36969
+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|725250|
|       Germany| 16694|
|          EIRE| 15743|
|        France| 13812|
|   Netherlands|  5088|
|         Spain|  3719|
|       Belgium|  3068|
|   Switzerland|  3011|
|      Portugal|  2446|
|     Australia|  1812|
+--------------+------+
only showing top 10 rows



In [18]:
check.describe(
    "Quantity",
    "Price",
    "TotalAmount"
).show()

+-------+------------------+------------------+------------------+
|summary|          Quantity|             Price|       TotalAmount|
+-------+------------------+------------------+------------------+
|  count|            805549|            805549|            805549|
|   mean|13.290522364250965| 3.206561485397269|22.026505103971285|
| stddev|143.63408833154656|29.199172655405402| 224.0419275415367|
|    min|                 1|             0.001|               0.0|
|    max|             80995|           10953.5|          168469.6|
+-------+------------------+------------------+------------------+



In [19]:
spark.stop()